# 02 — Data Cleaning Pipeline

**Objective:** Clean the raw datasets through imputation, outlier treatment, deduplication, and validation. Produce clean datasets for downstream analysis.

---

In [15]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.config import *
from src.utils import iqr_outlier_bounds, cap_outliers, save_csv

pd.set_option('display.max_columns', None)

In [16]:
# Load raw data
loyalty = pd.read_csv(RAW_DATA_DIR / LOYALTY_HISTORY_FILE)
flights = pd.read_csv(RAW_DATA_DIR / FLIGHT_ACTIVITY_FILE)

print(f'Loyalty: {loyalty.shape}  |  Flights: {flights.shape}')
print(f'\nLoyalty missing:\n{loyalty.isnull().sum()[loyalty.isnull().sum() > 0]}')

Loyalty: (16737, 16)  |  Flights: (392936, 8)

Loyalty missing:
Salary                 4238
Cancellation Year     14670
Cancellation Month    14670
dtype: int64


## 1. Missing Value Imputation

### 1.1 Numerical Imputation — Salary

**Strategy:** Median imputation grouped by `Education × Province`. Salary is correlated with education level and geographic region, so grouped imputation is more accurate than global median.

In [17]:
salary_missing_before = loyalty['Salary'].isnull().sum()
salary_missing_pct = salary_missing_before / len(loyalty) * 100
print(f'Salary missing BEFORE: {salary_missing_before:,} ({salary_missing_pct:.1f}%)')

# Create Salary_Missing indicator (useful for modeling if missingness is informative)
loyalty['Salary_Missing'] = loyalty['Salary'].isnull().astype(int)

# Grouped median imputation: Education × Province
loyalty['Salary'] = loyalty.groupby(['Education', 'Province'])['Salary'].transform(
    lambda x: x.fillna(x.median())
)

# Fallback: if still missing (rare group combinations), use Education-level median
loyalty['Salary'] = loyalty.groupby('Education')['Salary'].transform(
    lambda x: x.fillna(x.median())
)

# Final fallback: global median
loyalty['Salary'] = loyalty['Salary'].fillna(loyalty['Salary'].median())

salary_missing_after = loyalty['Salary'].isnull().sum()
print(f'Salary missing AFTER:  {salary_missing_after:,}')

# Visualize before/after
fig = px.histogram(loyalty, x='Salary', nbins=50, title='Salary Distribution (After Imputation)',
                   color_discrete_sequence=['#636EFA'])
fig.update_layout(template='plotly_dark', height=350)
fig.show()

Salary missing BEFORE: 4,238 (25.3%)
Salary missing AFTER:  0


### 1.2 Categorical Imputation

**Strategy:** Cancellation Year/Month are null for active members — this is intentional and not truly "missing". No imputation needed for these columns.

In [18]:
# Check all remaining missing values
remaining_missing = loyalty.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print('Remaining missing values:')
print(remaining_missing)
print('\nNote: Cancellation Year/Month are intentionally null for active members.')

Remaining missing values:
Cancellation Year     14670
Cancellation Month    14670
dtype: int64

Note: Cancellation Year/Month are intentionally null for active members.


## 2. Outlier Treatment

### 2.1 IQR Detection + Percentile Capping

In [19]:
# === Loyalty History: Salary & CLV ===
print('=== OUTLIER TREATMENT: Loyalty History ===')

for col in ['Salary', 'CLV']:
    before_stats = loyalty[col].describe()
    lower, upper = iqr_outlier_bounds(loyalty[col])
    outliers_before = ((loyalty[col] < lower) | (loyalty[col] > upper)).sum()
    
    # Apply percentile capping
    loyalty[col] = cap_outliers(loyalty[col], PERCENTILE_LOWER, PERCENTILE_UPPER)
    
    after_stats = loyalty[col].describe()
    lower_new, upper_new = iqr_outlier_bounds(loyalty[col])
    outliers_after = ((loyalty[col] < lower_new) | (loyalty[col] > upper_new)).sum()
    
    print(f'\n  {col}:')
    print(f'    Before: min={before_stats["min"]:.0f}, max={before_stats["max"]:.0f}, outliers={outliers_before:,}')
    print(f'    After:  min={after_stats["min"]:.0f}, max={after_stats["max"]:.0f}, outliers={outliers_after:,}')

=== OUTLIER TREATMENT: Loyalty History ===

  Salary:
    Before: min=-58486, max=407228, outliers=877
    After:  min=43788, max=239867, outliers=787

  CLV:
    Before: min=1898, max=83325, outliers=1,485
    After:  min=2230, max=35929, outliers=1,485


In [20]:
# === Flight Activity: Cap extreme values ===
print('\n=== OUTLIER TREATMENT: Flight Activity ===')

flight_num_cols = ['Total Flights', 'Distance', 'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed']

for col in flight_num_cols:
    outliers_before = ((flights[col] < iqr_outlier_bounds(flights[col])[0]) | 
                       (flights[col] > iqr_outlier_bounds(flights[col])[1])).sum()
    flights[col] = cap_outliers(flights[col], PERCENTILE_LOWER, PERCENTILE_UPPER)
    outliers_after = ((flights[col] < iqr_outlier_bounds(flights[col])[0]) | 
                      (flights[col] > iqr_outlier_bounds(flights[col])[1])).sum()
    print(f'  {col}: {outliers_before:,} → {outliers_after:,} outliers')


=== OUTLIER TREATMENT: Flight Activity ===
  Total Flights: 8,911 → 8,911 outliers
  Distance: 21,903 → 21,903 outliers
  Points Accumulated: 22,338 → 22,338 outliers
  Points Redeemed: 23,885 → 23,885 outliers
  Dollar Cost Points Redeemed: 23,885 → 23,885 outliers


In [21]:
# Visualize distributions after capping
fig = make_subplots(rows=1, cols=2, subplot_titles=['Salary (Capped)', 'CLV (Capped)'])
fig.add_trace(go.Histogram(x=loyalty['Salary'], nbinsx=50, marker_color='#636EFA', name='Salary'), row=1, col=1)
fig.add_trace(go.Histogram(x=loyalty['CLV'], nbinsx=50, marker_color='#EF553B', name='CLV'), row=1, col=2)
fig.update_layout(template='plotly_dark', height=350, title='Distributions After Outlier Capping', showlegend=False)
fig.show()

## 3. Duplicate Removal

In [22]:
print('=== DUPLICATE REMOVAL ===')

# --- Loyalty History ---
exact_dupes_loyalty = loyalty.duplicated().sum()
print(f'\nLoyalty History:')
print(f'  Exact duplicates: {exact_dupes_loyalty:,}')
loyalty = loyalty.drop_duplicates()

# Customer-level duplicates (keep first occurrence)
customer_dupes = loyalty[PK].duplicated().sum()
print(f'  Customer-level duplicates: {customer_dupes:,}')
if customer_dupes > 0:
    loyalty = loyalty.drop_duplicates(subset=[PK], keep='first')
    print(f'  After dedup: {loyalty.shape[0]:,} rows')

# --- Flight Activity ---
exact_dupes_flights = flights.duplicated().sum()
print(f'\nFlight Activity:')
print(f'  Exact duplicates: {exact_dupes_flights:,}')
flights = flights.drop_duplicates()

# Composite key duplicates
composite_dupes = flights.duplicated(subset=[PK, 'Year', 'Month']).sum()
print(f'  Composite key duplicates (Loyalty Number, Year, Month): {composite_dupes:,}')
if composite_dupes > 0:
    # Aggregate duplicates: sum flights/distance/points
    flights = flights.groupby([PK, 'Year', 'Month'], as_index=False).agg({
        'Total Flights': 'sum',
        'Distance': 'sum',
        'Points Accumulated': 'sum',
        'Points Redeemed': 'sum',
        'Dollar Cost Points Redeemed': 'sum',
    })
    print(f'  After aggregation: {flights.shape[0]:,} rows')

print(f'\nFinal shapes: Loyalty={loyalty.shape}  Flights={flights.shape}')

=== DUPLICATE REMOVAL ===

Loyalty History:
  Exact duplicates: 0
  Customer-level duplicates: 0

Flight Activity:
  Exact duplicates: 1,923
  Composite key duplicates (Loyalty Number, Year, Month): 1,948
  After aggregation: 389,065 rows

Final shapes: Loyalty=(16737, 17)  Flights=(389065, 8)


## 4. Create Derived Fields

In [23]:
# Add useful derived fields to Loyalty History

# Enrollment date
loyalty['Enrollment_Date'] = pd.to_datetime(
    loyalty['Enrollment Year'].astype(int).astype(str) + '-' + 
    loyalty['Enrollment Month'].astype(int).astype(str) + '-01'
)

# Cancellation date (NaT for active members)
loyalty['Cancellation_Date'] = pd.NaT
mask = loyalty['Cancellation Year'].notna()
loyalty.loc[mask, 'Cancellation_Date'] = pd.to_datetime(
    loyalty.loc[mask, 'Cancellation Year'].astype(int).astype(str) + '-' + 
    loyalty.loc[mask, 'Cancellation Month'].astype(int).astype(str) + '-01'
)

# Is cancelled flag
loyalty['Is_Cancelled'] = loyalty['Cancellation Year'].notna().astype(int)

# Tenure in months (from enrollment to cancellation or to max date in data)
max_date = pd.Timestamp('2018-12-01')  # End of observation period
loyalty['End_Date'] = loyalty['Cancellation_Date'].fillna(max_date)
loyalty['Tenure_Months'] = ((loyalty['End_Date'] - loyalty['Enrollment_Date']).dt.days / 30.44).round(0).astype(int)

# Education ordinal encoding
education_map = {v: i for i, v in enumerate(EDUCATION_ORDER)}
loyalty['Education_Ordinal'] = loyalty['Education'].map(education_map)

# Loyalty card ordinal encoding
card_map = {v: i for i, v in enumerate(LOYALTY_CARD_ORDER)}
loyalty['Loyalty_Card_Ordinal'] = loyalty['Loyalty Card'].map(card_map)

print('Derived fields added:')
print('  Enrollment_Date, Cancellation_Date, Is_Cancelled')
print('  Tenure_Months, Education_Ordinal, Loyalty_Card_Ordinal')
print(f'\nTenure stats:\n{loyalty["Tenure_Months"].describe()}')

Derived fields added:
  Enrollment_Date, Cancellation_Date, Is_Cancelled
  Tenure_Months, Education_Ordinal, Loyalty_Card_Ordinal

Tenure stats:
count    16737.000000
mean        35.451933
std         24.384892
min          0.000000
25%         11.000000
50%         33.000000
75%         57.000000
max         80.000000
Name: Tenure_Months, dtype: float64


In [24]:
# Add YearMonth to flights for easier time-series operations
flights['YearMonth'] = pd.to_datetime(
    flights['Year'].astype(str) + '-' + flights['Month'].astype(str).str.zfill(2) + '-01'
)

print(f'Flight date range: {flights["YearMonth"].min()} to {flights["YearMonth"].max()}')
print(f'Unique months: {flights["YearMonth"].nunique()}')

Flight date range: 2017-01-01 00:00:00 to 2018-12-01 00:00:00
Unique months: 24


## 6. Save Cleaned Datasets

In [25]:
# Save cleaned datasets
save_csv(loyalty, CLEANED_DATA_DIR / 'loyalty_cleaned.csv')
save_csv(flights, CLEANED_DATA_DIR / 'flights_cleaned.csv')

print(f'\nCleaned Loyalty: {loyalty.shape}')
print(f'Cleaned Flights: {flights.shape}')
print(f'\nFiles saved to: {CLEANED_DATA_DIR}')

2026-06-11 22:54:29 | airline_loyalty | INFO | Saved: loyalty_cleaned.csv (16,737 rows × 24 cols)
2026-06-11 22:54:29 | airline_loyalty | INFO | Saved: flights_cleaned.csv (389,065 rows × 9 cols)



Cleaned Loyalty: (16737, 24)
Cleaned Flights: (389065, 9)

Files saved to: C:\Users\admin\OneDrive\Desktop\airline-loyality\data\cleaned
